# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maqd1/maqd-ml-lab/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [6]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.550
Precision@50:  hand rule 0.680   vs   tree 0.600


Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [7]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [8]:

# Experiment 1: Increase the decision tree depth

for depth in [2, 3, 4]:
    experiment_tree = DecisionTreeClassifier(
        max_depth=depth,
        class_weight="balanced",
        random_state=42
    )

    experiment_tree.fit(X, y)

    experiment_score = experiment_tree.predict_proba(X)[:, 1]

    print(f"\nTree depth: {depth}")
    print(
        "Precision@50:",
        round(precision_at_k(experiment_score, y, 50), 3)
    )

    print(export_text(experiment_tree, feature_names=features))



Tree depth: 2
Precision@50: 0.6
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0


Tree depth: 3
Precision@50: 0.72
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- avg_position <= 25.15
|   |   |   |--- class: 0
|   |   |--- avg_position >  25.15
|   |   

In [9]:

# Experiment 2: Change the available features

new_features = [
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "word_count",
    "engagement_rate"
]

X_new = (
    df[new_features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

new_tree = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
)

new_tree.fit(X_new, y)

print(export_text(new_tree, feature_names=new_features))


|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- class: 0
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- class: 0



In [10]:
print(df["engagement_rate"].describe())

print("\nAverage engagement rate by label:")
print(
    df.groupby("is_declining_label")["engagement_rate"]
    .mean()
)

count    30000.000000
mean         2.534520
std          8.310096
min          0.000000
25%          0.000000
50%          0.000000
75%          1.350000
max        100.000000
Name: engagement_rate, dtype: float64

Average engagement rate by label:
is_declining_label
0    2.649728
1    2.437193
Name: engagement_rate, dtype: float64


In [11]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label', 'hand_rule_score']


In [12]:
print("Number of clients:", df["client_id"].nunique())

print("\nPages per client:")
print(df["client_id"].value_counts().describe())

Number of clients: 32

Pages per client:
count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000
Name: count, dtype: float64


In [13]:
client_sizes = (
    df.groupby("client_id")
    .agg(
        page_count=("content_id", "count"),
        declining_rate=("is_declining_label", "mean")
    )
    .sort_values("page_count", ascending=False)
)

print(client_sizes.to_string())

                   page_count  declining_rate
client_id                                    
client_19581e27de        7008        0.490154
client_6208ef0f77        3681        0.643847
client_4e07408562        2294        0.493461
client_3fdba35f04        2267        0.838553
client_f369cb89fc        1796        0.601336
client_8527a891e2        1194        0.437186
client_a88a7902cb        1171        0.667805
client_d4735e3a26        1106        0.219711
client_7f2253d7e2        1043        0.936721
client_f74efabef1        1031        0.542192
client_d029fa3a95         952        0.552521
client_349c41201b         763        0.689384
client_e629fa6598         720        0.455556
client_e29c9c180c         708        0.086158
client_2c624232cd         649        0.194145
client_8722616204         578        0.669550
client_4ec9599fc2         556        0.595324
client_bbb965ab0c         505        0.548515
client_25fc0e7096         476        0.000000
client_624b60c58c         337     

In [14]:
from pathlib import Path

script_path = Path("scripts/03_train_model.py")

print(script_path.read_text())

from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

from ml_utils import (
    MODEL_CATEGORICAL_FEATURES,
    MODEL_NUMERIC_FEATURES,
    OUTPUT_DIR,
    PROCESSED_DIR,
    display_path,
    precision_at_k,
    write_json,
)


FEATURE_PATH = PROCESSED_DIR / "refresh_feature_vector.csv"
BASELINE_PATH = PROCESSED_DIR / "baseline_refresh_queue.csv"
PREDICTION_PATH = PROCESSED_DIR / "model_predictions.csv"
RESULT_PATH = OUTPUT_DIR / "model_results.json"
RANDOM_STATE = 42


def parse_args

In [15]:

# Experiment 3A: Reproduce FlyRank's client-aware split

all_indices = np.arange(len(df))

client_series = df["client_id"].fillna("unknown").astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()

random_generator = np.random.default_rng(42)
shuffled_clients = random_generator.permutation(unique_clients)

test_client_count = max(
    1, int(round(len(shuffled_clients) * 0.2))
)

test_clients = set(shuffled_clients[:test_client_count])

test_mask = client_series.isin(test_clients).to_numpy()

train_indices = all_indices[~test_mask]
test_indices = all_indices[test_mask]

print("Selected test clients:", len(test_clients))
print("Training clients:", client_series.iloc[train_indices].nunique())
print("Testing clients:", client_series.iloc[test_indices].nunique())

print("\nTraining pages:", len(train_indices))
print("Testing pages:", len(test_indices))

print(
    "\nTraining declining rate:",
    round(y[train_indices].mean(), 3)
)

print(
    "Testing declining rate:",
    round(y[test_indices].mean(), 3)
)


Selected test clients: 6
Training clients: 26
Testing clients: 6

Training pages: 27675
Testing pages: 2325

Training declining rate: 0.555
Testing declining rate: 0.391


In [16]:
print("Training classes:")
print(np.unique(y[train_indices]))

print("\nTesting classes:")
print(np.unique(y[test_indices]))

train_clients = set(client_series.iloc[train_indices])
test_clients = set(client_series.iloc[test_indices])

print("\nClient overlap:", train_clients.intersection(test_clients))

Training classes:
[0 1]

Testing classes:
[0 1]

Client overlap: set()


In [17]:
from sklearn.tree import DecisionTreeClassifier

# 1. Rebuild the original six-feature dataset
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X_holdout = (
    df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

# 2. Train using only the training clients
tree_holdout = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
)

tree_holdout.fit(
    X_holdout.iloc[train_indices],
    y[train_indices]
)

# 3. Generate predictions for unseen testing clients
tree_test_scores = tree_holdout.predict_proba(
    X_holdout.iloc[test_indices]
)[:, 1]

# 4. Get hand-rule scores for those same testing clients
hand_rule_test_scores = (
    df["hand_rule_score"].to_numpy()[test_indices]
)

y_test = y[test_indices]

# 5. Compare both approaches
for k in (20, 50):
    print(
        f"Precision@{k}: "
        f"Hand rule = "
        f"{precision_at_k(hand_rule_test_scores, y_test, k):.3f} "
        f" | Tree = "
        f"{precision_at_k(tree_test_scores, y_test, k):.3f}"
    )

Precision@20: Hand rule = 0.300  | Tree = 0.450
Precision@50: Hand rule = 0.400  | Tree = 0.560


In [18]:
for depth in (2, 3, 4):
    model = DecisionTreeClassifier(
        max_depth=depth,
        class_weight="balanced",
        random_state=42
    )

    model.fit(
        X_holdout.iloc[train_indices],
        y[train_indices]
    )

    scores = model.predict_proba(
        X_holdout.iloc[test_indices]
    )[:, 1]

    print(f"\nTree depth: {depth}")

    for k in (20, 50):
        print(
            f"Precision@{k}: "
            f"{precision_at_k(scores, y_test, k):.3f}"
        )


Tree depth: 2
Precision@20: 0.450
Precision@50: 0.560

Tree depth: 3
Precision@20: 0.650
Precision@50: 0.600

Tree depth: 4
Precision@20: 0.600
Precision@50: 0.620


In [19]:
from sklearn.tree import export_text

depth3_tree = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42
)

depth3_tree.fit(
    X_holdout.iloc[train_indices],
    y[train_indices]
)

print(
    export_text(
        depth3_tree,
        feature_names=features
    )
)

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.90
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.90
|   |   |--- content_age_days <= 97.00
|   |   |   |--- class: 0
|   |   |--- content_age_days >  97.00
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- avg_position <= 25.15
|   |   |   |--- class: 0
|   |   |--- avg_position >  25.15
|   |   |   |--- class: 0



In [20]:
from sklearn.tree import export_text

# Identify the leaf reached by each training page
leaf_ids = depth3_tree.apply(
    X_holdout.iloc[train_indices]
)

# Combine leaf IDs with the actual labels
leaf_analysis = pd.DataFrame({
    "leaf_id": leaf_ids,
    "actual_label": y[train_indices]
})

# Summarise each leaf
summary = (
    leaf_analysis.groupby("leaf_id")
    .agg(
        page_count=("actual_label", "size"),
        declining_rate=("actual_label", "mean")
    )
    .reset_index()
)

summary["declining_rate"] = summary["declining_rate"].round(3)

print(summary.to_string(index=False))

 leaf_id  page_count  declining_rate
       3         932           0.004
       4          10           0.300
       6          80           0.462
       7        1270           0.229
      10       12219           0.695
      11        4158           0.568
      13        6558           0.518
      14        2448           0.314


In [21]:
test_leaf_ids = depth3_tree.apply(
    X_holdout.iloc[test_indices]
)

test_leaf_analysis = pd.DataFrame({
    "leaf_id": test_leaf_ids,
    "actual_label": y_test
})

test_leaf_summary = (
    test_leaf_analysis.groupby("leaf_id")
    .agg(
        page_count=("actual_label", "size"),
        declining_rate=("actual_label", "mean")
    )
    .reset_index()
)

test_leaf_summary["declining_rate"] = (
    test_leaf_summary["declining_rate"].round(3)
)

print(test_leaf_summary.to_string(index=False))

 leaf_id  page_count  declining_rate
       3         306           0.013
       4           5           0.200
       6          23           0.609
       7         420           0.210
      10         916           0.560
      11         523           0.434
      13         129           0.481
      14           3           0.000


# Experiment 3D — Comparing training and testing leaves

I compared the declining rates of the decision-tree leaves across training and unseen testing clients.
Some patterns remained relatively stable. For example, leaf 3 had declining rates of 0.4% during training
and 1.3% during testing, while leaf 7 had 22.9% and 21.0%, respectively.

However, leaf 10 decreased from 69.5% during training to 56.0% during testing.
This demonstrates that learned patterns may behave differently across unseen clients. Very small leaves also
require caution when interpreting their declining rates.

### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.